# Milestone 1: Data Preprocessing

In [1]:
import pandas as pd

try:

    Data = pd.read_csv("../data/raw/Vehicle-Health-Telemetry-Dataset.csv")
    print("CSV file loaded successfully.")

except FileNotFoundError:

    print("Error: data.csv not found. Please check the file path.")

except Exception as e:

    print(f"An error occurred: {e}")

CSV file loaded successfully.


In [2]:
Data.head(5)

,Timestamp,Vehicle_ID,Engine_RPM,Vehicle_Speed,Coolant_Temp,Oil_Pressure,Vibration_Z,Engine_Load,Fuel_Rate,Intake_Air_Temp,...,Throttle_Position,Ambient_Temp,Brake_Pressure,Acceleration_X,Acceleration_Y,Odometer,Engine_Run_Time,Vehicle_Status,Fault_Label,Fault_Type
0,2026-06-01 00:00:00,VH-001,0.0,0.0,NaN,1.661008,0.013402,0.0,0.0,21.933113,...,0.0,21.446732,0.5,0.0,0.0,0.0,0.0,OFF,0.0,Normal
1,2026-06-01 00:00:01,VH-001,0.0,0.0,21.259663,1.625691,0.007419,0.0,0.0,20.778847,...,0.0,21.256239,0.5,0.0,NaN,0.0,0.0,OFF,0.0,Normal
2,2026-06-01 00:00:02,VH-001,0.0,0.0,21.487158,0.650765,0.017300,0.0,0.0,22.161366,...,0.0,21.492025,0.5,0.0,0.0,0.0,0.0,OFF,0.0,Normal
3,2026-06-01 00:00:03,VH-001,0.0,0.0,21.700083,1.108323,0.014455,0.0,0.0,20.367332,...,0.0,21.754627,0.5,0.0,0.0,0.0,0.0,OFF,0.0,Normal
4,2026-06-01 00:00:04,VH-001,0.0,0.0,21.402119,3.790915,0.000000,0.0,0.0,22.893360,...,0.0,21.227472,0.5,0.0,0.0,0.0,0.0,OFF,0.0,Normal


In [3]:
print(Data.columns.tolist())

['Timestamp', 'Vehicle_ID', 'Engine_RPM', 'Vehicle_Speed', 'Coolant_Temp', 'Oil_Pressure', 'Vibration_Z', 'Engine_Load', 'Fuel_Rate', 'Intake_Air_Temp', 'Battery_Voltage', 'Throttle_Position', 'Ambient_Temp', 'Brake_Pressure', 'Acceleration_X', 'Acceleration_Y', 'Odometer', 'Engine_Run_Time', 'Vehicle_Status', 'Fault_Label', 'Fault_Type']


In [4]:
Data['Timestamp'] = pd.to_datetime(Data['Timestamp'])

Data.set_index('Timestamp', inplace=True)

cols_to_drop = ['Vehicle_Status', 'Odometer', 'Engine_Run_Time', 'Vehicle_ID']
Data.drop(columns=[col for col in cols_to_drop if col in Data.columns], inplace=True)


In [5]:
missing_values = (Data.isnull().sum().reset_index())

missing_values.columns = ['Feature', 'Missing Values']

display(missing_values)

,Feature,Missing Values
0,Engine_RPM,4760
1,Vehicle_Speed,4881
2,Coolant_Temp,4823
3,Oil_Pressure,4911
4,Vibration_Z,4912
5,Engine_Load,4945
6,Fuel_Rate,4986
7,Intake_Air_Temp,4858
8,Battery_Voltage,4918
9,Throttle_Position,5012


In [6]:

Data.index = pd.to_datetime(Data.index, errors='coerce')
Data = Data.sort_index()

numeric_cols = Data.select_dtypes(include='number').columns

Data[numeric_cols] = Data[numeric_cols].interpolate(method='time')

In [7]:
missing_values = (Data.isnull().sum().reset_index())

missing_values.columns = ['Feature', 'Missing Values']

display(missing_values)

,Feature,Missing Values
0,Engine_RPM,0
1,Vehicle_Speed,0
2,Coolant_Temp,1
3,Oil_Pressure,0
4,Vibration_Z,0
5,Engine_Load,0
6,Fuel_Rate,0
7,Intake_Air_Temp,0
8,Battery_Voltage,0
9,Throttle_Position,0


In [8]:
col = 'Coolant_Temp'

Data[col] = Data[col].ffill().bfill()

In [9]:

numeric_cols = Data.select_dtypes(include='number').columns

window = 5
alpha = 1 / window

Data[numeric_cols] = Data[numeric_cols].apply(lambda col: col.ewm(alpha=alpha, adjust=False).mean())

Data.head()

,Engine_RPM,Vehicle_Speed,Coolant_Temp,Oil_Pressure,Vibration_Z,Engine_Load,Fuel_Rate,Intake_Air_Temp,Battery_Voltage,Throttle_Position,Ambient_Temp,Brake_Pressure,Acceleration_X,Acceleration_Y,Fault_Label,Fault_Type
Timestamp,,,,,,,,,,,,,,,,
2026-06-01 00:00:00,0.0,0.0,21.259663,1.661008,0.013402,0.0,0.0,21.933113,12.488478,0.0,21.446732,0.5,0.0,0.0,0.0,Normal
2026-06-01 00:00:01,0.0,0.0,21.259663,1.653945,0.012205,0.0,0.0,21.702260,12.488286,0.0,21.408633,0.5,0.0,0.0,0.0,Normal
2026-06-01 00:00:02,0.0,0.0,21.305162,1.453309,0.013224,0.0,0.0,21.794081,12.490938,0.0,21.425312,0.5,0.0,0.0,0.0,Normal
2026-06-01 00:00:03,0.0,0.0,21.384146,1.384312,0.013470,0.0,0.0,21.508731,12.497744,0.0,21.491175,0.5,0.0,0.0,0.0,Normal
2026-06-01 00:00:04,0.0,0.0,21.387741,1.865632,0.010776,0.0,0.0,21.785657,12.496880,0.0,21.438434,0.5,0.0,0.0,0.0,Normal


In [13]:
Data.to_csv("../data/processed/Cleaned-Vehicle-Health-Telemetry-Dataset.csv", index=True)